In [2]:
# Strat Data - 7-July-2025
# Purpose: Contract Deliverable for Bonsai Informatica. This application will prompt the 
#          user for the name of a data set to load and will then allow the user to search
#          for a word in one of four positions in the data field. 
#          This application satisfies the following requirements from the customer:
#
# Application Requirements:
#     1. The application must be able to read a data set from a file where the format is 
#        consistent with the provided data record format. 
#     2. The application must be able to retrieve a record or records from the file based 
#        on provided search criteria. 
#     3. When multiple records are returned, they must be displayed in ascending order of 
#        the TLEN field in the record. 
#     4. When displaying a record, only the TLEN field and the data field should be displayed.  
#     5. When displaying a record, each field should be separated by a tab. 
#     6. When displaying the data field, each word should be separated by a comma followed by a space. 
#     7. When displaying a data record, an indication of whether the record is valid, based 
#        on the SHA-256 checksum in its validation field, should be provided as the final field displayed. 
#     8. The application must be able to accept a word with an associated position indicator to 
#        search for that word in the corresponding position in the data field. 
#     9. When searching, if the word is not found in the correct position, the application must 
#        indicate that the word was not found in the correct position. 
#    10. When searching, if the word is not found in the correct position, the application must 
#        indicate if there are any records where the word was found in a different position in 
#        the data field. 
#    11. The application must display all records containing relevant information separated by 
#        header information. 
#    12. The application must be optimized to minimize the time necessary to perform the search function. 
 
#
# This application assumes that the first line of each data file is a header line that will be
# skipped when creating valid records.
#
# This application supports the following data record format:
#
# Each data record has five pipe-delimited fields, with each field as follows: 
#     1. sequence – a zero-based numerical sequence indicating the position of the record 
#                   in the data file. 
#     2. TLEN – a positive integer representing the length of the data field including spaces. 
#     3. data – a list of four words separated by spaces
#     4. validation – a SHA-256 checksum of the three previous fields inclusive of field-separating 
#                     characters between the three fields.
#
#
# Version               Author         Date              Description
#######################################################################################################
#    2                  fjm            8-Jul-2025        Added consistency check in data file import
#    1                  fjm            7-Jul-2025        Initial Code
#######################################################################################################

##
## Import csv module to make working with the data file easier
##
import csv

##
## Configure DEBUG flag. True = print DEBUG information, False = be quiet...
##
DEBUG = True

##
## Create a Python class to hold the data from each record
##
class datarecord:
    ##
    ## Class constructor
    ##
    ## Parameters:
    ##     sequence - sequence number of the record in the data file
    ##     TLEN - Length of data
    ##     data - List of four words
    ##     validation - SHA-256 validation string for the first three
    ##                  fields in the record
    ##
    def __init__(self,sequence,TLEN,data,validation):
        self.sequence = sequence
        self.TLEN = TLEN
        self.data = data
        self.validation = validation


    ##
    ## printHeader - function to Print Header information for records
    ##
    def printHeader(self):
        print("TLEN\tdata\n")

              
    ##
    ## printRecord - function to Print the record in the manner the 
    ## application requires. TLEN and then the Data fields with information
    ## separated by tabs. The data field will be printed with commas and 
    ## spaces separating the individual words.
    ##
    def printRecord(self):
        out = self.data[0] + ", " + self.data[1] + ", " + self.data[2] + ", " + self.data[3]
        print(f"{self.TLEN}\t{out}")
        


    ##
    ## hashFunction - function to calculate the hash value for a data
    ## column in this specific datarecord object. This hash function 
    ## creates a value from the characters in one column of data
    ## and then returns the integer remainder of that value divided by 
    ## tableSize. This will facilitate searching for values in a specific
    ## data column.
    ##
    ## Parameters:
    ##      column - An integer value between 0 and 3 inclusive representing
    ##               one of the four columns in the data field.
    ##      tableSize - variable indicating the size of the Hash Table. This
    ##                  must be indicated as an positive integer
    ##
    ## Returns:
    ##      positive integer indicating the index into the Hash Table of 
    ##      size tableSize.
    ##
    def hashFunction(self,column,tableSize):
        source = self.data[column]
        simpleSum = 0

        for char in source:
            simpleSum = simpleSum + ord(char)

        # Return the value adjusted for the correct Hash Table size
        return simpleSum % tableSize

    ##
    ## extHashIndex - function to calculate the hash value for this provided
    ## string. This hash function creates a value from the
    ## characters in the data parameter and then returns the integer
    ## remainder of that value divided by tableSize. This will facilitate 
    ## searching for values in a given sized Hash Table.
    ##
    ## Parameters:
    ##      data - The string to be hashed
    ##      tableSize - variable indicating the size of the Hash Table. This
    ##                  must be indicated as an positive integer
    ##
    ## Returns:
    ##      positive integer indicating the index into the Hash Table of 
    ##      size tableSize.
    ##
    def extHashIndex(data,tableSize):
        source = data
        simpleSum = 0

        for char in source:
            simpleSum = simpleSum + ord(char)

        return simpleSum % tableSize

        
    ##
    ## readDataFile - function to read a pipe delimited data file and return
    ## a list of datarecord object. This method assumes that the first line of
    ## the file is a header line.
    ##
    ## Parameters:
    ##     fileName - the name of the datafile to read from
    ##
    ## Returns:
    ##     list of datarecord objects in the same order as they appeared in
    ##     the data file.
    def readDataFile(fileName):
        ## define our output list
        records = []

        ## read data file
        with open(fileName,'r') as file:
            csvReader = csv.reader(file, delimiter='|')

            ## get the headers - this is used as a no-op as the headers
            ## aren't used here.
            header = next(csvReader)



            ## Read the file and create datarecord objects, add them to the list
            for row in csvReader:
                data = row[2].split()

                if len(data) != 4:
                    pass
                else:
                    records.append(datarecord(row[0],row[1],data,row[3]))

            ## close file
            file.close()

        return records

        
    ##
    ## matches - function to determine whether or not this record matches
    ## a specific search criteria.
    ##
    ## Parameters:
    ##      column - integer value to indicate which column to match against.
    ##               this selects which of the four columns in the data field
    ##               to test against
    ##      value - the value to test against
    ## 
    ## Returns:
    ##      True if the record match is successful, False if the record
    ##      does not match the search criteria
    def matches(self,column,value):
        return self.data[column] == value
            

##
## promptUser - function used to prompt the user for input. They will either
## enter a one word search string or a two word command to quit. This will
## also prompt the user for which column they wish to match against.
##
## Returns:
##      String containing user input
##      Integer between 0 and 3 inclusive indicating which column to 
##      match against.
##
def promptUser():
    print("")
    print("*************************************************************")
    print("")
    print("Please enter a single word to be located within the data set.")
    print("If you wish to quit, please enter: I quit\n")
    out = input("Search string: ")

    print("Please enter a column number from 0 to 3 inclusive indicating")
    print("your choice of column for the search.")
    column = int(input("Column Number: "))

    return out, column

##
## conductSearch - function used to conduct the records search. This function
## will first search the indicated data column and if no records are present 
## that match the search criteria, will then search the remaining data columns.
##
## Parameters:
##      search - the string to be located
##      column - the initial data column to search
##      aHT - The first hash table
##      bHT - The second hash table
##      cHT - The third hash table
##      dHT - The fourth hash table
##
## Returns:
##      this function returns no data, but will display formatted results to the screen.
##
def conductSearch(search, column, aHT, bHT, cHT, dHT):
    global DEBUG
    
    ## Calculate index - because all four hash tables are the same size, we only
    ## need to do this once
    index = datarecord.extHashIndex(search,len(aHT))

    ## Flag to determine if we need to check additional Hash Tables at all.
    checkAlt = False
    
    ## accumulator variable for how many records were checked.
    checked = 0

    ## display list
    matchingRecords = [ [], [], [], [] ]

    primary = column
    alternate = []

    ## Set alternate columns
    for i in range(4):
        if i != column:
            alternate.append(i)

    ## set bucket default
    bucket = aHT[index]
    
    ## Determine if we need to change the primary bucket assignment
    if primary == 1:
        bucket = bHT[index]
    elif primary == 2:
        bucket = cHT[index]
    elif primary == 3:
        bucket = dHT[index]

    ## Primary bucket checks
    if len(bucket) == 0:
        ## Nothing Here, check alternate Hash Table
        checkAlt = True
    else:
        tmp = []

        if DEBUG:
            pass
            ## print(f"Bucket: {bucket}")
            
        for i in bucket:
            checked = checked + 1
            if i.matches(column,search):
                tmp.append(i)
        if len(tmp) == 0:
            ## Nothing in primary, check alternate Hash Table
            checkAlt = True
        else:
            matchingRecords[primary] = tmp

    ## If True, check alternate tables - make sure not to re-process the 
    ## primary table
    if checkAlt:

        ## Loop through alternate Columns
        for j in alternate:
            if DEBUG:
                pass
                ## print(f"Alternate Column: {j}")

            ## Set appropriate bucket
            if j == 0:
                bucket = aHT[index]
            elif j == 1:
                bucket = bHT[index]
            elif j == 2:
                bucket = cHT[index]
            elif j == 3:
                bucket = dHT[index]
                
            if len(bucket) == 0:
                ## Nothing Here, continue with the next table
                pass
            else:
                tmp = []
                for i in bucket:
                    checked = checked + 1
                    if i.matches(j,search):
                        tmp.append(i)
                if len(tmp) == 0:
                    ## No matches, continue
                    pass
                else:
                    matchingRecords[j] = tmp

    ## if DEBUG:
    ##    print(matchingRecords[0])
    ##    print(matchingRecords[1])
    ##    print(matchingRecords[2])
    ##    print(matchingRecords[3])

    ## All checks have been completed, so we need to output our results.
    if not checkAlt:
        print(f"\n{len(matchingRecords[primary])} matching records were found in data")
        matchingRecords[primary][0].printHeader()
        for i in matchingRecords[primary]:
            i.printRecord()
    else:
        ## This is where it gets slightly more complicated.
        print(f"\nNo matching records for search term: {search} were found in data for column: {primary}")

        print(f"Checking for {search} in remaining columns produced the following results:")

        ## Column 0 - aHT
        if 0 != primary:
            if len(matchingRecords[0]) == 0:
                print(f"\nNo matching records were found in column 0")
            else:
                print(f"\n{len(matchingRecords[0])} matching records were found in column 0")
                matchingRecords[0][0].printHeader()
                for i in matchingRecords[0]:
                    i.printRecord()
                    
        ## Column 1 - bHT
        if 1 != primary:
            if len(matchingRecords[1]) == 0:
                print(f"\nNo matching records were found in column 1")
            else:
                print(f"\n{len(matchingRecords[1])} matching records were found in column 1")
                matchingRecords[1][0].printHeader()
                for i in matchingRecords[1]:
                    i.printRecord()
                    
        ## Column 2 - cHT
        if 2 != primary:
            if len(matchingRecords[2]) == 0:
                print(f"\nNo matching records were found in column 2")
            else:
                print(f"\n{len(matchingRecords[1])} matching records were found in column 2")
                matchingRecords[2][0].printHeader()
                for i in matchingRecords[2]:
                    i.printRecord()
            
        ## Column 3 - cHT
        if 3 != primary:
            if len(matchingRecords[3]) == 0:
                print(f"\nNo matching records were found in column 3")
            else:
                print(f"\n{len(matchingRecords[3])} matching records were found in column 3")
                matchingRecords[3][0].printHeader()
                for i in matchingRecords[3]:
                    i.printRecord()
                    
    ## Provide additional data
    print(f"\n{checked} records were checked while executing this search.")        
        

## 
## main program
##
if __name__ == '__main__':


    ##
    ## Get name of data file from user
    ## 
    dataFile = input("Please enter the name of the data file to process: ")

    if DEBUG:
        ## Print the name of the data file used
        print(f"File containing data set: {dataFile}")

    ##
    ## Read records into a list
    ##
    records = datarecord.readDataFile(dataFile)

    if DEBUG:
        ## Print number of records retrieved
        print(f"{len(records)} records retrieved.\n")

    ##
    ## Create Empty Hash Tables.
    ##
    aHT = []
    bHT = []
    cHT = []
    dHT = []

    ## Hash Table Size
    size = 10000
    
    ## Iterate to build hash table of length size.
    for i in range(size):
        tmpA = []
        tmpB = []
        tmpC = []
        tmpD = []
        
        aHT.append(tmpA)
        bHT.append(tmpB)
        cHT.append(tmpC)
        dHT.append(tmpD)

    if DEBUG:
        ## Print size of Hash Table
        print(f"a - Hash Table size = {len(aHT)} elements.")
        print(f"b - Hash Table size = {len(bHT)} elements.")
        print(f"c - Hash Table size = {len(cHT)} elements.")
        print(f"d - Hash Table size = {len(dHT)} elements.")

        ## Print Hash Table
        ## print("Hash Table - D")
        ## print(dHT)

    ## 
    ## Populate the Hash Tables to promote fast searching
    ## 
    difference = 0
    for i in records:
        aIndex = i.hashFunction(0,size)
        aHT[aIndex].append(i)

        bIndex = i.hashFunction(1,size)
        bHT[bIndex].append(i)
                    
        cIndex = i.hashFunction(2,size)
        cHT[cIndex].append(i)

        dIndex = i.hashFunction(3,size)
        dHT[dIndex].append(i)
        
        ## if DEBUG:
        ##    print(f"A Index: {aIndex}, B Index: {bIndex}, C Index: {cIndex}, D Index: {dIndex}") 

        if aIndex != bIndex or bIndex != cIndex or cIndex != dIndex:
            difference = difference + 1
            
        if DEBUG:
            pass
            ## Print Entry
            ## i.printRecord()

    if DEBUG:
        print(f"{difference} different hash table indexes across {len(records)} records.")
        ## Calculate the number of empty buckets and the bucket with the largest size
        empty = 0
        maxLen = 0
        for i in range(size):
            entries = len(aHT[i])
            if entries == 0:
                empty = empty + 1
            elif entries > maxLen:
                maxLen = entries
        print("\nHash Table - A")
        print(f"Empty buckets: {empty} out of {size} in Hash Table A")
        print(f"Maximum bucket size: {maxLen} in Hash Table A")

        empty = 0
        maxLen = 0
        for i in range(size):
            entries = len(bHT[i])
            if entries == 0:
                empty = empty + 1
            elif entries > maxLen:
                maxLen = entries
        print("\nHash Table - B")
        print(f"Empty buckets: {empty} out of {size} in Hash Table B")
        print(f"Maximum bucket size: {maxLen} in Hash Table B")

        empty = 0
        maxLen = 0
        for i in range(size):
            entries = len(cHT[i])
            if entries == 0:
                empty = empty + 1
            elif entries > maxLen:
                maxLen = entries
        print("\nHash Table - C")
        print(f"Empty buckets: {empty} out of {size} in Hash Table C")
        print(f"Maximum bucket size: {maxLen} in Hash Table C")

        empty = 0
        maxLen = 0
        for i in range(size):
            entries = len(dHT[i])
            if entries == 0:
                empty = empty + 1
            elif entries > maxLen:
                maxLen = entries
        print("\nHash Table - D")
        print(f"Empty buckets: {empty} out of {size} in Hash Table D")
        print(f"Maximum bucket size: {maxLen} in Hash Table D")
        

    ##
    ## Main loopI q
    ##
    exit = False
    while not exit:
        search, column = promptUser()

        ## Check for exit condition
        if len(search.split()) != 1:
            exit = True
            print("Exiting...")
        else:
            ## Run search
            print(f"Searching for {search} in column {column}...")
            conductSearch(search,column,aHT,bHT,cHT,dHT)

Please enter the name of the data file to process:  data/set1.csv


File containing data set: data/set1.csv
1000 records retrieved.

a - Hash Table size = 10000 elements.
b - Hash Table size = 10000 elements.
c - Hash Table size = 10000 elements.
d - Hash Table size = 10000 elements.
1000 different hash table indexes across 1000 records.

Hash Table - A
Empty buckets: 9401 out of 10000 in Hash Table A
Maximum bucket size: 6 in Hash Table A

Hash Table - B
Empty buckets: 9404 out of 10000 in Hash Table B
Maximum bucket size: 6 in Hash Table B

Hash Table - C
Empty buckets: 9390 out of 10000 in Hash Table C
Maximum bucket size: 6 in Hash Table C

Hash Table - D
Empty buckets: 9399 out of 10000 in Hash Table D
Maximum bucket size: 7 in Hash Table D

*************************************************************

Please enter a single word to be located within the data set.
If you wish to quit, please enter: I quit



Search string:  test


Please enter a column number from 0 to 3 inclusive indicating
your choice of column for the search.


Column Number:  adsf


ValueError: invalid literal for int() with base 10: 'adsf'